# Java vs Python — Evaluation Engine Comparison

Runs both engines against **test.openml.org** and compares their outputs.
**Java is the original/source of truth; the Python port is configured to match it.**

Prerequisites:
- Python side: the `EvaluationEnginePython` project (invoked via `uv` by the cells below).
- Java side: build the jar once — `cd EvaluationEngine && mvn clean package -DskipTests` — then set `JAVA_JAR`.
- A **test.openml.org API key** (`TEST_API_KEY`) is required for the paths that upload (`evaluate_run`, `process_dataset`). Read-only paths (`generate_folds`, `merge_datasets`) work without one.

The notebook kernel only needs the Python standard library (it shells out to `uv` / `java`); the read-back helpers use `urllib` + `xml.etree`, no extra packages.


In [1]:
# ---- EDIT THESE -----------------------------------------------------------
JAVA_JAR = "/Users/smukherjee/Documents/Projects/OpenML/EvaluationEngine/target/EvaluationEngine-1.2.2-SNAPSHOT-jar-with-dependencies.jar"
PY_PROJECT = "/Users/smukherjee/Documents/Projects/OpenML/EvaluationEnginePython"
TEST_SERVER = "https://test.openml.org/"
TEST_API_KEY = ""        # paste your test.openml.org API key (needed for uploads)

# Common ids on test.openml.org (tweak as needed):
TASK_ID = 119              # generate_folds: task 1 = 10-fold CV on dataset 1 (anneal)
RUN_ID = 6              # evaluate_run: a run id that exists on test with predictions
DATASET_ID = 20           # process_dataset: dataset id to extract features for
MERGE_TASK_ID = ""       # merge_datasets: a MultiTask task id on test (set a real one)
DATA_FORMAT = "arff"      # Python dataset format: "arff" (default) or "parquet" — passed to the port via `-df`
# --------------------------------------------------------------------------


## Helpers
Run-once utility cells: shell out to each engine and compare outputs.

In [2]:
import subprocess, pathlib, difflib
import urllib.request, xml.etree.ElementTree as ET

NS = {"oml": "http://openml.org/openml"}
_QUOTES = "'" + '"'        # both quote chars, for stripping nominal values

def _cfg(api_key=None):
    return f"api_key={api_key if api_key is not None else TEST_API_KEY};server={TEST_SERVER}"

def run_java(fn, *opts, api_key=None):
    jar = pathlib.Path(JAVA_JAR)
    if not jar.exists():
        print(f"!! jar not found: {JAVA_JAR}")
        print("   build it: cd <EvaluationEngine> && mvn clean package -DskipTests")
        return None
    cmd = ["java", "-jar", str(jar), "-config", _cfg(api_key), "-f", fn, *map(str, opts)]
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout.strip(): print("[java stdout]", r.stdout.strip()[-1200:])
    if r.stderr.strip(): print("[java stderr]", r.stderr.strip()[-1200:])
    return r

def run_py(fn, *opts, test=True):
    cmd = ["uv", "run", "--project", PY_PROJECT, "python", "-m", "main",
           "-f", fn, "-df", DATA_FORMAT, *(["-test"] if test else []),
           *map(str, opts)]
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=PY_PROJECT)
    if r.stdout.strip(): print("[py stdout]", r.stdout.strip()[-1200:])
    if r.stderr.strip(): print("[py stderr]", r.stderr.strip()[-1200:])
    return r

# ---- splits ARFF comparison (robust to column order / quoting) ------------
def parse_splits(path):
    attrs, rows, in_data = [], set(), False
    for line in pathlib.Path(path).read_text().splitlines():
        s, up = line.strip(), line.strip().upper()
        if up.startswith("@ATTRIBUTE"):
            attrs.append(s.split(None, 2)[1].strip(_QUOTES).lower())
        elif up == "@DATA":
            in_data = True
        elif in_data and s and not s.startswith("%"):
            vals = [x.strip().strip(_QUOTES) for x in s.split(",")]
            row = dict(zip(attrs, vals))
            rows.add((row.get("type"), row.get("rowid"), row.get("repeat"),
                      row.get("fold"), row.get("sample")))
    return rows

def compare_splits(java_path, py_path):
    if not (pathlib.Path(java_path).exists() and pathlib.Path(py_path).exists()):
        print("missing output file(s)"); return
    j, p = parse_splits(java_path), parse_splits(py_path)
    oj, op = j - p, p - j
    print(f"java rows={len(j)}  py rows={len(p)}  only_java={len(oj)}  only_py={len(op)}")
    if oj: print("sample java-only:", sorted(oj)[:5])
    if op: print("sample py-only:", sorted(op)[:5])
    print("=> MATCH" if not oj and not op else "=> MISMATCH")

def diff_files(a, b, la="java", lb="python", n=40):
    if not (pathlib.Path(a).exists() and pathlib.Path(b).exists()):
        print("missing output file(s)"); return
    xa = pathlib.Path(a).read_text().splitlines()
    xb = pathlib.Path(b).read_text().splitlines()
    d = list(difflib.unified_diff(xa, xb, fromfile=la, tofile=lb, n=1))
    print(f"{la}: {len(xa)} lines   {lb}: {len(xb)} lines")
    print("IDENTICAL" if not d else f"DIFFER ({len(d)} diff lines), first {n}:")
    for line in d[:n]: print(line)

def arff_data_rows(path):
    rows, in_data = [], False
    for line in pathlib.Path(path).read_text().splitlines():
        s = line.strip()
        if s.upper() == "@DATA": in_data = True
        elif in_data and s and not s.startswith("%"):
            rows.append(tuple(x.strip().strip(_QUOTES) for x in s.split(",")))
    return rows

# ---- server read-back (stdlib) -------------------------------------------
def _get_xml(url):
    with urllib.request.urlopen(url) as r:
        return ET.fromstring(r.read())

def _parse_array(s):
    if not s:
        return None
    return [float(x) for x in s.strip("[]").split(",") if x.strip() != ""]

def fetch_evals(run_id):
    """{function: {"value": ..., "array_data": ...}} as stored on test.

    The /evaluation/list/run endpoint returns one row per metric (the global
    aggregate), with per-fold values in ``array_data``."""
    root = _get_xml(f"{TEST_SERVER}api/v1/xml/evaluation/list/run/{run_id}")
    out = {}
    for e in root.findall("oml:evaluation", NS):
        out[e.findtext("oml:function", "", NS)] = {
            "value": e.findtext("oml:value", None, NS),
            "array_data": e.findtext("oml:array_data", None, NS),
        }
    return out

def evals_diff(j, p, label=""):
    jn, pn = set(j), set(p)
    print(f"=== {label}: {len(jn)} java metrics, {len(pn)} py metrics; names match: {jn == pn} ===")
    if jn - pn: print("  only java:", sorted(jn - pn))
    if pn - jn: print("  only py :", sorted(pn - jn))
    diffs = []
    for name in sorted(jn & pn):
        jv, pv = j[name]["value"], p[name]["value"]
        ja, pa = _parse_array(j[name]["array_data"]), _parse_array(p[name]["array_data"])
        differ = False
        try:
            if abs(float(jv) - float(pv)) > 1e-9:
                differ = True
        except (TypeError, ValueError):
            if jv != pv:
                differ = True
        if ja is not None or pa is not None:
            if ja != pa and (ja is None or pa is None or len(ja) != len(pa)
                             or any(abs(a - b) > 1e-9 for a, b in zip(ja, pa))):
                differ = True
        if differ:
            diffs.append(name)
            print(f"  {name}: value java={jv} py={pv} | array java={j[name]['array_data']} py={p[name]['array_data']}")
    print("=> all metrics MATCH" if not diffs and jn == pn else f"=> {len(diffs)} metric(s) differ")

def fetch_features(did):
    return _features_from(_get_xml(f"{TEST_SERVER}api/v1/xml/data/features/{did}"))

def parse_features_xml(xml_str):
    return _features_from(ET.fromstring(xml_str.strip().encode()))

def _features_from(root):
    out = {}
    for f in root.findall("oml:feature", NS):
        name = f.findtext("oml:name", "", NS)
        fields, nominals = {}, []
        for c in f:
            local = c.tag.split("}", 1)[-1].removeprefix("oml:")
            if local == "nominal_value":
                nominals.append(c.text)
            else:
                fields[local] = c.text
        fields["nominal_value"] = nominals
        out[name] = fields
    return out

def features_diff(j, p):
    jn, pn = set(j), set(p)
    print(f"feature names match: {jn == pn}  (only_java={len(jn-pn)}, only_py={len(pn-jn)})")
    if jn - pn: print("  only java:", sorted(jn - pn))
    if pn - jn: print("  only py :", sorted(pn - jn))
    ndiff = 0
    for name in sorted(jn & pn):
        for key in sorted(set(j[name]) | set(p[name])):
            jv, pv = j[name].get(key), p[name].get(key)
            if key == "nominal_value":
                jv, pv = sorted(x for x in (jv or []) if x), sorted(x for x in (pv or []) if x)
            if jv != pv:
                ndiff += 1; print(f"  {name}.{key}: java={jv!r}  py={pv!r}")
    print("=> features MATCH" if ndiff == 0 and jn == pn else f"=> {ndiff} feature field(s) differ")

print("helpers loaded")


helpers loaded


## 1. generate_folds  *(task id → splits ARFF)*
Both take `-id <task_id>`; folds/repeats/type come from the task's estimation procedure, seed 0 (Java `Main.FOLD_GENERATION_SEED`).

In [3]:
# Java (source of truth)
run_java("generate_folds", "-id", TASK_ID, "-o", "/tmp/java_folds.arff")

$ java -jar /Users/smukherjee/Documents/Projects/OpenML/EvaluationEngine/target/EvaluationEngine-1.2.2-SNAPSHOT-jar-with-dependencies.jar -config api_key=;server=https://test.openml.org/ -f generate_folds -id 119 -o /tmp/java_folds.arff
[java stderr]  2026 1:05:32 PM org.apache.http.client.protocol.ResponseProcessCookies processCookies
Jul 29, 2026 1:05:32 PM org.apache.http.client.protocol.ResponseProcessCookies processCookies
Jul 29, 2026 1:05:32 PM org.apache.http.client.protocol.ResponseProcessCookies processCookies
Jul 29, 2026 1:05:33 PM org.apache.http.client.protocol.ResponseProcessCookies processCookies


CompletedProcess(args=['java', '-jar', '/Users/smukherjee/Documents/Projects/OpenML/EvaluationEngine/target/EvaluationEngine-1.2.2-SNAPSHOT-jar-with-dependencies.jar', '-config', 'api_key=;server=https://test.openml.org/', '-f', 'generate_folds', '-id', '119', '-o', '/tmp/java_folds.arff'], returncode=0, stdout='', stderr='WARNING: A terminally deprecated method in sun.misc.Unsafe has been called\nWARNING: sun.misc.Unsafe::objectFieldOffset has been called by com.thoughtworks.xstream.converters.reflection.SunUnsafeReflectionProvider (file:/Users/smukherjee/Documents/Projects/OpenML/EvaluationEngine/target/EvaluationEngine-1.2.2-SNAPSHOT-jar-with-dependencies.jar)\nWARNING: Please consider reporting this to the maintainers of class com.thoughtworks.xstream.converters.reflection.SunUnsafeReflectionProvider\nWARNING: sun.misc.Unsafe::objectFieldOffset will be removed in a future release\nJul 29, 2026 1:05:32 PM org.apache.http.client.protocol.ResponseProcessCookies processCookies\nWARNING

In [4]:
# Python (configured to match Java: task id, task-derived procedure, seed 0)
run_py("generate_folds", "-id", TASK_ID, "-o", "/tmp/py_folds.arff")

$ uv run --project /Users/smukherjee/Documents/Projects/OpenML/EvaluationEnginePython python -m main -f generate_folds -test -id 119 -o /tmp/py_folds.arff
[py stdout] wrote 768 split rows to /tmp/py_folds.arff


CompletedProcess(args=['uv', 'run', '--project', '/Users/smukherjee/Documents/Projects/OpenML/EvaluationEnginePython', 'python', '-m', 'main', '-f', 'generate_folds', '-test', '-id', '119', '-o', '/tmp/py_folds.arff'], returncode=0, stdout='wrote 768 split rows to /tmp/py_folds.arff\n', stderr='')

In [5]:
# Compare split assignments (TRAIN/TEST per rowid x repeat x fold),
# ignoring ARFF formatting differences.
compare_splits("/tmp/java_folds.arff", "/tmp/py_folds.arff")

java rows=768  py rows=768  only_java=365  only_py=365
sample java-only: [('TEST', '106', '0', '0', None), ('TEST', '107', '0', '0', None), ('TEST', '108', '0', '0', None), ('TEST', '11', '0', '0', None), ('TEST', '110', '0', '0', None)]
sample py-only: [('TEST', '1', '0', '0', None), ('TEST', '109', '0', '0', None), ('TEST', '118', '0', '0', None), ('TEST', '12', '0', '0', None), ('TEST', '122', '0', '0', None)]
=> MISMATCH


## 2. evaluate_run  *(run id → metric scores; uploads to test)*
Both compute metrics for the run's predictions and upload to test. We read the stored evaluations back and compare per metric. **Note:** `kb_relative_information_score` is flagged in the port as possibly diverging from Weka.

In [22]:
# Java evaluates + uploads. (Needs TEST_API_KEY with write access.)
run_java("evaluate_run", "-id", RUN_ID)
JAVA_EVALS = fetch_evals(RUN_ID)
print("java evals:", len(JAVA_EVALS))

$ java -jar /Users/smukherjee/Documents/Projects/OpenML/EvaluationEngine/target/EvaluationEngine-1.2.2-SNAPSHOT-jar-with-dependencies.jar -config api_key=;server=https://test.openml.org/ -f evaluate_run -id 6
[java stderr] ead>
<body>
  <div class="container">
    <h1>Not found <span>:(</span></h1>
    <p>Sorry, but the page you were trying to view does not exist.</p>
    <p>It looks like this was the result of either:</p>
    <ul>
      <li>a mistyped address</li>
      <li>an out-of-date link</li>
    </ul>
    <script>
      var GOOG_FIXURL_LANG = (navigator.language || '').slice(0,2),GOOG_FIXURL_SITE = location.host;
    </script>
    <script src="http://linkhelp.clients.google.com/tbproxy/lh/wm/fixurl.js"></script>
  </div>
</body>
</html>

[29-07-2026 12:27:46] [OK] [Process Run] Start uploading results ... 
[29-07-2026 12:27:46] [ERROR] [Process Run] An error occured during API call: Api key not set. 
java.lang.Exception: Api key not set. 
	at org.openml.apiconnector.io.HttpConn

In [23]:
# Python evaluates + uploads (overwrites the same run on test).
run_py("evaluate_run", "-id", RUN_ID)
PY_EVALS = fetch_evals(RUN_ID)
print("py evals:", len(PY_EVALS))

$ uv run --project /Users/smukherjee/Documents/Projects/OpenML/EvaluationEnginePython python -m src.main -f evaluate_run -test -id 6
[py stderr] Error: 412 Client Error: Precondition Failed for url: https://test.openml.org/api/v1/run/evaluate
py evals: 18


In [24]:
evals_diff(JAVA_EVALS, PY_EVALS, label="evaluate_run")

=== evaluate_run: 18 java metrics, 18 py metrics; names match: True ===
=> all metrics MATCH


## 3. process_dataset  *(dataset id → feature XML)*
Java `process_dataset` uploads features to test; Python has a read-only `process_dataset_print` that prints the same XML. We compare Python's printed features against the features Java stored on test.

In [27]:
# Python: extract features locally, print XML (no upload, no key needed).
rpy = run_py("process_dataset_print", "-id", DATASET_ID)
PY_FEAT = parse_features_xml(rpy.stdout) if rpy else {}
print("py features:", len(PY_FEAT))

$ uv run --project /Users/smukherjee/Documents/Projects/OpenML/EvaluationEnginePython python -m src.main -f process_dataset_print -test -id 20
[py stdout] number_of_integer_values>768</oml:number_of_integer_values>
		<oml:number_of_real_values>0</oml:number_of_real_values>
		<oml:number_of_values>768</oml:number_of_values>
		<oml:maximum_value>81.0</oml:maximum_value>
		<oml:minimum_value>21.0</oml:minimum_value>
		<oml:mean_value>33.240885416666664</oml:mean_value>
		<oml:standard_deviation>11.752572645994181</oml:standard_deviation>
	</oml:feature>
	<oml:feature>
		<oml:index>8</oml:index>
		<oml:name>class</oml:name>
		<oml:data_type>nominal</oml:data_type>
		<oml:nominal_value>tested_negative</oml:nominal_value>
		<oml:nominal_value>tested_positive</oml:nominal_value>
		<oml:is_target>true</oml:is_target>
		<oml:is_ignore>false</oml:is_ignore>
		<oml:is_row_identifier>false</oml:is_row_identifier>
		<oml:number_of_missing_values>0</oml:number_of_missing_values>
		<oml:number_of_dis

In [28]:
# Java: extract + upload features to test. (Needs TEST_API_KEY.)
run_java("process_dataset", "-id", DATASET_ID)
JAVA_FEAT = fetch_features(DATASET_ID)
print("java features:", len(JAVA_FEAT))

$ java -jar /Users/smukherjee/Documents/Projects/OpenML/EvaluationEngine/target/EvaluationEngine-1.2.2-SNAPSHOT-jar-with-dependencies.jar -config api_key=;server=https://test.openml.org/ -f process_dataset -id 20
[java stderr] ava:79)
	at org.openml.apiconnector.io.OpenmlBasicConnector.dataFeaturesUpload(OpenmlBasicConnector.java:356)
	at org.openml.apiconnector.io.OpenmlConnector.dataFeaturesUpload(OpenmlConnector.java:77)
	at org.openml.webapplication.ProcessDataset.process(ProcessDataset.java:74)
	at org.openml.webapplication.ProcessDataset.<init>(ProcessDataset.java:33)
	at org.openml.webapplication.Main.main(Main.java:117)
[29-07-2026 12:28:46] [Error] [Process Dataset] Error while processing dataset. Marking this in database.
java.lang.Exception: Api key not set. 
	at org.openml.apiconnector.io.HttpConnector.doApiPostRequest(HttpConnector.java:79)
	at org.openml.apiconnector.io.OpenmlBasicConnector.dataFeaturesUpload(OpenmlBasicConnector.java:356)
	at org.openml.apiconnector.io.O

In [29]:
features_diff(JAVA_FEAT, PY_FEAT)

feature names match: True  (only_java=0, only_py=0)
  age.maximum_value: java=None  py='81.0'
  age.mean_value: java=None  py='33.240885416666664'
  age.minimum_value: java=None  py='21.0'
  age.number_of_distinct_values: java=None  py='52'
  age.number_of_integer_values: java=None  py='768'
  age.number_of_real_values: java=None  py='0'
  age.number_of_unique_values: java=None  py='5'
  age.number_of_values: java=None  py='768'
  age.standard_deviation: java=None  py='11.752572645994181'
  class.class_distribution: java=None  py='tested_negative:500,tested_positive:268'
  class.number_of_distinct_values: java=None  py='2'
  class.number_of_nominal_values: java=None  py='768'
  class.number_of_unique_values: java=None  py='0'
  class.number_of_values: java=None  py='768'
  insu.maximum_value: java=None  py='846.0'
  insu.mean_value: java=None  py='79.79947916666667'
  insu.minimum_value: java=None  py='0.0'
  insu.number_of_distinct_values: java=None  py='186'
  insu.number_of_integer_

## 4. merge_datasets  *(MultiTask task id → merged ARFF)*
Set `MERGE_TASK_ID` to a real MultiTask task id on test. Both write the merged ARFF to a file.

In [30]:
assert MERGE_TASK_ID, "set MERGE_TASK_ID to a real MultiTask task id on test"
run_java("merge_datasets", "-id", MERGE_TASK_ID, "-o", "/tmp/java_merged.arff")

AssertionError: set MERGE_TASK_ID to a real MultiTask task id on test

In [ ]:
run_py("merge_datasets", "-id", MERGE_TASK_ID, "-o", "/tmp/py_merged.arff")

In [ ]:
# Merged ARFFs differ cosmetically (Weka vs liac-arff); compare row sets.
diff_files("/tmp/java_merged.arff", "/tmp/py_merged.arff")
jr, pr = set(arff_data_rows("/tmp/java_merged.arff")), set(arff_data_rows("/tmp/py_merged.arff"))
print(f"java data rows={len(jr)}  py data rows={len(pr)}  only_java={len(jr-pr)} only_py={len(pr-jr)}")
print("=> row sets MATCH" if jr == pr else "=> row sets DIFFER")

## 5. Not yet ported to Python

`all_wrong`, `different_predictions` (Java: `InstanceBased.java`) and `challenge` (Java: `ChallengeSets.java`) are **NotImplemented** in the Python port — each pair below runs the Java command and confirms the Python side raises `NotImplementedError`.


### all_wrong  *(needs `-r` run ids + `-t` task id)*

In [ ]:
# Java
run_java("all_wrong", "-r", "1,2", "-t", str(TASK_ID))

In [ ]:
# Python — NotImplemented
run_py("all_wrong", "-r", "1,2", "-t", str(TASK_ID))

### different_predictions  *(needs `-r` run ids + `-t` task id)*

In [ ]:
# Java
run_java("different_predictions", "-r", "1,2", "-t", str(TASK_ID))

In [ ]:
# Python — NotImplemented
run_py("different_predictions", "-r", "1,2", "-t", str(TASK_ID))

### challenge  *(needs `-t` task id + `-mode train|test`)*

In [ ]:
# Java
run_java("challenge", "-t", str(TASK_ID), "-mode", "train")

In [ ]:
# Python — NotImplemented
run_py("challenge", "-t", str(TASK_ID), "-mode", "train")